In [1]:
from sentence_transformers import SentenceTransformer
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, classification_report
import pandas as pd
import numpy as np

In [2]:
train_df = pd.read_csv("imdb_train_clean.csv")
test_df = pd.read_csv("imdb_test_clean.csv")

print(train_df.head)
print(test_df.head)


print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

<bound method NDFrame.head of            id                                           sentence  label  \
0           1  I really don't understand why people get so up...      1   
1           1  Remember folks, this is an SNL movie, not anyt...      1   
2           1  The Ladies Man is a hilarious movie, albeit st...      1   
3           1  Yes some of the jokes are stupid, and yes, the...      1   
4           1  I really don't understand how anyone couldn't ...      1   
...       ...                                                ...    ...   
261888  24999  I can only describe it as a 1990s equivalent t...      0   
261889  25000  Good films cannot solely be based on a beautif...      0   
261890  25000            Surprised to see it has won two awards.      0   
261891  25000  I first saw that kind of films from China, vis...      0   
261892  25000               This is not one of them, I'm afraid.      0   

                                           sentence_clean  
0       I

In [3]:
def remove_single_sentence_docs(df):
    # Count number of sentences per document
    doc_counts = df.groupby("id").size()

    # Keep only document IDs with more than 1 sentence
    valid_ids = doc_counts[doc_counts > 4].index

    # Filter dataframe
    return df[df["id"].isin(valid_ids)].reset_index(drop=True)


train_df = remove_single_sentence_docs(train_df)
test_df  = remove_single_sentence_docs(test_df)


print("Train min sentences:", train_df.groupby("id").size().min())
print("Test min sentences:", test_df.groupby("id").size().min())



print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train min sentences: 5
Test min sentences: 5
Train shape: (250763, 4)
Test shape: (245383, 4)


In [4]:
def sample_balanced_documents_sequential(
    df,
    id_col="id",
    label_col="label",
    n_docs_per_class=500,
):
    """
    Sequentially samples documents (not sentences) so that:
    - Documents are balanced across labels
    - All sentences for a document are kept
    - Order is preserved (no random sampling)
    """

    sampled_dfs = []

    for label in [0, 1]:
        # Get document ids in original order
        doc_ids = (
            df[df[label_col] == label]
            .groupby(id_col)
            .size()
            .index
        )

        # Take first N document IDs sequentially
        selected_doc_ids = doc_ids[:n_docs_per_class]

        # Keep ALL sentences for selected documents
        sampled_label_df = df[df[id_col].isin(selected_doc_ids)]
        sampled_dfs.append(sampled_label_df)

    balanced_df = pd.concat(sampled_dfs).reset_index(drop=True)

    return balanced_df

train_small = sample_balanced_documents_sequential(
    train_df,
    id_col="id",
    label_col="label",
    n_docs_per_class=5500
)

test_small = sample_balanced_documents_sequential(
    test_df,
    id_col="id",
    label_col="label",
    n_docs_per_class=5500
)

print("Train shape:", train_small.shape)
print("Test shape:", test_small.shape)

print("Train label counts:")
print(train_small.groupby("label")["id"].nunique())

print("Train min sentences per doc:",
      train_small.groupby("id").size().min())

print("Test min sentences per doc:",
      test_small.groupby("id").size().min())





Train shape: (129648, 4)
Test shape: (127969, 4)
Train label counts:
label
0    5500
1    5500
Name: id, dtype: int64
Train min sentences per doc: 5
Test min sentences per doc: 5


In [5]:
print("Train min sentences:", train_small.groupby("id").size().min())
print("Test min sentences:", test_small.groupby("id").size().min())

Train min sentences: 5
Test min sentences: 5


In [6]:
model = SentenceTransformer("bert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [7]:
def encode_sentences(df):
    embeddings = model.encode(
        df["sentence_clean"].tolist(),
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True
    )
    return embeddings

train_embeddings = encode_sentences(train_small)
test_embeddings  = encode_sentences(test_small)

Batches:   0%|          | 0/4052 [00:00<?, ?it/s]

Batches:   0%|          | 0/4000 [00:00<?, ?it/s]

In [8]:
svm = SVC(
    kernel="rbf",
    probability=True,
    class_weight="balanced",   # <-- IMPORTANT
    random_state=42
)
svm.fit(train_embeddings, train_small["label"])

SVC(class_weight='balanced', probability=True, random_state=42)

In [9]:
train_small["svm_prob"] = svm.predict_proba(train_embeddings)[:, 1]
test_small["svm_prob"]  = svm.predict_proba(test_embeddings)[:, 1]

In [10]:
def build_doc_vectors(df, prob_col="svm_prob"):
    grouped = df.groupby("id")[prob_col].apply(list)
    min_len = grouped.apply(len).min()

    X = np.array([
        vals[:min_len] for vals in grouped
    ])
    return X, grouped.index.tolist(), min_len

In [11]:
X_train, train_ids, min_len = build_doc_vectors(train_small)
X_test, test_ids, _ = build_doc_vectors(test_small)

print(f"Truncated to {min_len} sentences per document")

Truncated to 5 sentences per document


In [12]:
def get_doc_labels(df, doc_ids): # Use integer IDs to match vectors
  return df.groupby("id")["label"].first().values
  # return df.groupby("id_int")["Status"].first().loc[doc_i


y_train = get_doc_labels(train_small, train_ids)
y_test  = get_doc_labels(test_small, test_ids)

In [13]:
doc_clf = LogisticRegression(class_weight="balanced",max_iter=1000)
doc_clf.fit(X_train, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000)

In [14]:
def evaluate(model, X, y, name="SET"):
    preds = model.predict(X)
    probs = model.predict_proba(X)[:, 1]

    print(f"\n===== {name} RESULTS =====")
    print("Accuracy:", accuracy_score(y, preds))
    print("F1 Score:", f1_score(y, preds))
    print("MCC:", matthews_corrcoef(y, preds))
    print("\nClassification Report:\n", classification_report(y, preds))


evaluate(doc_clf, X_train, y_train, "TRAIN")
evaluate(doc_clf, X_test, y_test, "TEST")


===== TRAIN RESULTS =====
Accuracy: 0.8844545454545455
F1 Score: 0.8846956363966252
MCC: 0.7689158141858515

Classification Report:
               precision    recall  f1-score   support

           0       0.89      0.88      0.88      5500
           1       0.88      0.89      0.88      5500

    accuracy                           0.88     11000
   macro avg       0.88      0.88      0.88     11000
weighted avg       0.88      0.88      0.88     11000


===== TEST RESULTS =====
Accuracy: 0.8280909090909091
F1 Score: 0.826050961273112
MCC: 0.6563623806438502

Classification Report:
               precision    recall  f1-score   support

           0       0.82      0.84      0.83      5500
           1       0.84      0.82      0.83      5500

    accuracy                           0.83     11000
   macro avg       0.83      0.83      0.83     11000
weighted avg       0.83      0.83      0.83     11000

